In [ ]:
!pip install tensorflow pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [ ]:
dataset = pd.read_csv("/content/Aggregrated_Data.csv")
dataset.head()

,timestamp,satellite_prn,satellite_elevation_deg,satellite_azimuth_deg,temperature_C,humidity_%,rain_intensity_mm_hr,rain_sensor_value,snr_measured_dbhz,label
0,2024-01-01 00:00,8.23,44.94,313.18,34.99,58.64,0.00,0.00,51,0
1,2024-01-01 00:00,2.94,22.20,200.38,33.96,55.17,0.57,0.00,47,0
2,2024-01-01 00:00,11.30,26.44,162.51,32.92,50.66,0.00,0.00,42,0
3,2024-01-01 00:00,26.70,77.68,58.32,31.32,40.02,1.36,0.00,44,0
4,2024-01-01 00:00,21.89,82.50,54.38,28.65,89.37,2.79,49.88,39,1


In [ ]:
dataset = dataset[
    (dataset["satellite_elevation_deg"] >= 10) &
    (dataset["satellite_elevation_deg"] <= 70) &
    (dataset["satellite_azimuth_deg"] >= 0) &
    (dataset["satellite_azimuth_deg"] <= 360) &
    (dataset["satellite_prn"] >= 1) &
    (dataset["satellite_prn"] <= 32)
]

In [ ]:
print(dataset.isnull().sum())

timestamp                  0
satellite_prn              0
satellite_elevation_deg    0
satellite_azimuth_deg      0
temperature_C              0
humidity_%                 0
rain_intensity_mm_hr       0
rain_sensor_value          0
snr_measured_dbhz          0
label                      0
dtype: int64


In [ ]:
(dataset == 0).sum()

,0
timestamp,0
satellite_prn,0
satellite_elevation_deg,0
satellite_azimuth_deg,659
temperature_C,618
humidity_%,629
rain_intensity_mm_hr,4119
rain_sensor_value,4359
snr_measured_dbhz,0
label,8146


In [ ]:
columns = [
    "satellite_prn",
    "satellite_elevation_deg",
    "satellite_azimuth_deg",
    "temperature_C",
    "humidity_%",
    "rain_sensor_value",
    "snr_measured_dbhz"
]

for col in columns:
    mean_val = dataset.loc[dataset[col] != 0, col].mean()
    dataset[col] = dataset[col].replace(0, mean_val)

In [ ]:
X = dataset[
    [
    "satellite_prn",
    "satellite_elevation_deg",
    "satellite_azimuth_deg",
    "temperature_C",
    "humidity_%",
    "rain_sensor_value",
    "snr_measured_dbhz"
    ]
]

y = dataset["label"]

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = tf.keras.Sequential([

    tf.keras.layers.Dense(
        8,
        activation="relu",
        input_shape=(7,)
    ),

    tf.keras.layers.Dense(
        3,
        activation="softmax"
    )

])

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/30
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.4600 - loss: 1.2510 - val_accuracy: 0.9613 - val_loss: 0.3049
Epoch 2/30
407/407 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9760 - loss: 0.2266 - val_accuracy: 0.9886 - val_loss: 0.1014
Epoch 3/30
407/407 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9910 - loss: 0.0865 - val_accuracy: 0.9895 - val_loss: 0.0596
Epoch 4/30
407/407 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9913 - loss: 0.0545 - val_accuracy: 0.9899 - val_loss: 0.0436
Epoch 5/30
407/407 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9926 - loss: 0.0420 - val_accuracy: 0.9902 - val_loss: 0.0354
Epoch 6/30
407/407 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9911 - loss: 0.0352 - val_accuracy: 0.9908 - val_loss: 0.0306
Epoch 7/30
407/407 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9925 - loss: 0.0298 - val_accuracy: 0.9914 - val_loss: 0.0272
Epoch 8/30
407/407 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9926 - loss: 0.0279 - val_accuracy: 0.

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Accuracy:", accuracy)

128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9963 - loss: 0.0127
Accuracy: 0.9943433403968811


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("rain_model.tflite", "wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmp2_4o6_9r'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 7), dtype=tf.float32, name='keras_tensor_3')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  135368947264080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135368947264848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135368947264272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135368947265040: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [ ]:
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_quant_model = converter.convert()

with open("rain_model_quant.tflite", "wb") as f:
    f.write(tflite_quant_model)

Saved artifact at '/tmp/tmp1nirril9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 7), dtype=tf.float32, name='keras_tensor_3')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  135368947264080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135368947264848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135368947264272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135368947265040: TensorSpec(shape=(), dtype=tf.resource, name=None)
